# Notebook 02

# Silver Hazard Layer

## Objective

Transform the Bronze rainfall dataset into an analytical spatial dataset by creating a master ERA5 grid with unique grid identifiers.


In [2]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

BRONZE_FILE = (
    PROJECT_ROOT
    / "data"
    / "bronze"
    / "bronze_rainfall_2023.parquet"
)

SILVER_DIR = (
    PROJECT_ROOT
    / "data"
    / "silver"
)

print(BRONZE_FILE.exists())

True


In [4]:
bronze = pd.read_parquet(BRONZE_FILE)

bronze.head()

,date,latitude,longitude,number,rainfall_m,rainfall_mm
0,2023-01-01,38.0,68.00,0,0.0,0.0
1,2023-01-01,38.0,68.25,0,0.0,0.0
2,2023-01-01,38.0,68.50,0,0.0,0.0
3,2023-01-01,38.0,68.75,0,0.0,0.0
4,2023-01-01,38.0,69.00,0,0.0,0.0


In [5]:
# ============================================================
# Create Grid Master
# ============================================================

grid_master = (
    bronze[["latitude", "longitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
    .reset_index(drop=True)
)

grid_master.head()

,latitude,longitude
0,6.0,68.00
1,6.0,68.25
2,6.0,68.50
3,6.0,68.75
4,6.0,69.00


In [6]:
print(grid_master.shape)

(15609, 2)


In [7]:
grid_master.duplicated().sum()

np.int64(0)

In [8]:
# ============================================================
# Assign Grid ID
# ============================================================

grid_master.insert(0, "grid_id", range(1, len(grid_master) + 1))

grid_master.head()

,grid_id,latitude,longitude
0,1,6.0,68.00
1,2,6.0,68.25
2,3,6.0,68.50
3,4,6.0,68.75
4,5,6.0,69.00


In [9]:
print("Minimum Grid ID:", grid_master["grid_id"].min())
print("Maximum Grid ID:", grid_master["grid_id"].max())
print("Unique Grid IDs:", grid_master["grid_id"].nunique())

Minimum Grid ID: 1
Maximum Grid ID: 15609
Unique Grid IDs: 15609


In [10]:
# ============================================================
# Attach Grid ID to Bronze Dataset
# ============================================================

silver = bronze.merge(
    grid_master,
    on=["latitude", "longitude"],
    how="left"
)

silver.head()

,date,latitude,longitude,number,rainfall_m,rainfall_mm,grid_id
0,2023-01-01,38.0,68.00,0,0.0,0.0,15489
1,2023-01-01,38.0,68.25,0,0.0,0.0,15490
2,2023-01-01,38.0,68.50,0,0.0,0.0,15491
3,2023-01-01,38.0,68.75,0,0.0,0.0,15492
4,2023-01-01,38.0,69.00,0,0.0,0.0,15493


In [11]:
print("Rows:", silver.shape[0])
print("Columns:", silver.shape[1])

print("\nMissing Grid IDs:", silver["grid_id"].isna().sum())

Rows: 5697285
Columns: 7

Missing Grid IDs: 0


In [17]:
# ============================================================
# Export Grid Master
# ============================================================

grid_master_file = (
    SILVER_DIR /
    "silver_grid_master.parquet"
)

grid_master.to_parquet(
    grid_master_file,
    index=False
)

print("Grid Master exported successfully!")
print(grid_master_file)

Grid Master exported successfully!
d:\Work\Projects\India_Flood_Risk_Platform\data\silver\silver_grid_master.parquet


In [18]:
# ============================================================
# Export Silver Rainfall Dataset
# ============================================================

silver_file = (
    SILVER_DIR /
    "silver_rainfall_2023.parquet"
)

silver.to_parquet(
    silver_file,
    index=False
)

print("Silver Rainfall exported successfully!")
print(silver_file)

Silver Rainfall exported successfully!
d:\Work\Projects\India_Flood_Risk_Platform\data\silver\silver_rainfall_2023.parquet


In [19]:
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)

print(f"Grid Cells          : {len(grid_master):,}")
print(f"Rainfall Records    : {len(silver):,}")
print(f"Unique Grid IDs     : {silver['grid_id'].nunique():,}")
print(f"Date Range          : {silver['date'].min()} to {silver['date'].max()}")
print(f"Missing Grid IDs    : {silver['grid_id'].isna().sum()}")

print("\nOutputs")
print(f"Grid Master : {grid_master_file}")
print(f"Rainfall    : {silver_file}")

SILVER LAYER SUMMARY
Grid Cells          : 15,609
Rainfall Records    : 5,697,285
Unique Grid IDs     : 15,609
Date Range          : 2023-01-01 00:00:00 to 2023-12-31 00:00:00
Missing Grid IDs    : 0

Outputs
Grid Master : d:\Work\Projects\India_Flood_Risk_Platform\data\silver\silver_grid_master.parquet
Rainfall    : d:\Work\Projects\India_Flood_Risk_Platform\data\silver\silver_rainfall_2023.parquet


In [20]:
# ============================================================
# Create Final Silver Rainfall Table
# ============================================================

silver_final = silver[
    [
        "date",
        "grid_id",
        "rainfall_mm"
    ]
].copy()

silver_final.head()

,date,grid_id,rainfall_mm
0,2023-01-01,15489,0.0
1,2023-01-01,15490,0.0
2,2023-01-01,15491,0.0
3,2023-01-01,15492,0.0
4,2023-01-01,15493,0.0


In [21]:
# ============================================================
# Create Final Silver Rainfall Table
# ============================================================

silver_final = silver[
    [
        "date",
        "grid_id",
        "rainfall_mm"
    ]
].copy()

silver_final.head()

,date,grid_id,rainfall_mm
0,2023-01-01,15489,0.0
1,2023-01-01,15490,0.0
2,2023-01-01,15491,0.0
3,2023-01-01,15492,0.0
4,2023-01-01,15493,0.0
